# Proyecto SQL: Análisis de una plataforma de libros

## Introducción

Durante la pandemia de COVID-19, el consumo de contenido digital aumentó significativamente. Una empresa emergente dedicada a los amantes de los libros desea desarrollar una nueva aplicación y necesita comprender mejor el comportamiento de lectura de sus usuarios.                    
                                       
La base de datos disponible contiene información sobre:                                         
                                     
Libros                       
Autores                                     
Editoriales                                    
Calificaciones                               
Reseñas                                          
                                                
El objetivo es obtener información que permita identificar tendencias relevantes para el diseño del nuevo producto.

### Librerias

In [20]:
import pandas as pd
from sqlalchemy import create_engine
import matplotlib.pyplot as plt

### Conexión a la base de datos

In [21]:
db_config = {
    'user': 'practicum_student',
    'pwd': 's65BlTKV3faNIGhmvJVzOqhs',
    'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
    'port': 6432,
    'db': 'data-analyst-final-project-db'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine (
    connection_string,
    connect_args={'sslmode': 'require'}
)

### Exploración inicial de las tablas
Books

In [22]:
query = """
SELECT *
FROM books
LIMIT 5;
"""

pd.read_sql(query, engine)

,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268


Authors

In [23]:
query = """
SELECT *
FROM authors
LIMIT 5;
"""

pd.read_sql(query, engine)

,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd


Publishers

In [24]:
query = """
SELECT *
FROM publishers
LIMIT 5;
"""

pd.read_sql(query, engine)

,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company


Ratings

In [25]:
query = """
SELECT *
FROM ratings
LIMIT 5;
"""

pd.read_sql(query, engine)

,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2


Reviews

In [26]:
query = """
SELECT *
FROM reviews
LIMIT 5;
"""

pd.read_sql(query, engine)

,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


## Objetivo 1
Encontrar el número de libros publicados después del 1 de enero del año 2000
### Consulta SQL

In [27]:
query = """
SELECT COUNT(*) AS total_books
FROM books
WHERE publication_date > '2000-01-01';
"""

books_after_2000 = pd.read_sql(query, engine)

books_after_2000

,total_books
0,819


Conclusión

In [28]:
print(
    f"Se encontraron {books_after_2000.iloc[0,0]} libros "
    "publicados después del 1 de enero de 2000."
)

Se encontraron 819 libros publicados después del 1 de enero de 2000.


## Objetivo 2
Encontrar el número de reseñas y la calificación promedio para cada libro
### Consulta SQL

In [29]:
query = """
SELECT
    b.book_id,
    b.title,
    COUNT(DISTINCT rev.review_id) AS review_count,
    ROUND(AVG(r.rating),2) AS average_rating
FROM books b
LEFT JOIN reviews rev
    ON b.book_id = rev.book_id
LEFT JOIN ratings r
    ON b.book_id = r.book_id
GROUP BY
    b.book_id,
    b.title
ORDER BY review_count DESC;
"""

book_reviews_ratings = pd.read_sql(query, engine)

book_reviews_ratings.head()

,book_id,title,review_count,average_rating
0,948,Twilight (Twilight #1),7,3.66
1,963,Water for Elephants,6,3.98
2,734,The Glass Castle,6,4.21
3,302,Harry Potter and the Prisoner of Azkaban (Harr...,6,4.41
4,695,The Curious Incident of the Dog in the Night-Time,6,4.08


Conclusión

In [30]:
book_reviews_ratings.describe()

,book_id,review_count,average_rating
count,1000.000000,1000.000000,1000.000000
mean,500.500000,2.793000,3.899040
std,288.819436,1.074852,0.562388
min,1.000000,0.000000,1.500000
25%,250.750000,2.000000,3.500000
50%,500.500000,3.000000,4.000000
75%,750.250000,3.000000,4.330000
max,1000.000000,7.000000,5.000000


#### Interpretación:

Se observa qué libros generan mayor participación.                           
Los libros con muchas reseñas suelen generar más interacción.                 
La calificación promedio permite identificar los títulos mejor valorados.

## Objetivo 3
Editorial con el mayor número de libros de más de 50 páginas
### Consulta SQL

In [31]:
query = """
SELECT
    p.publisher,
    COUNT(b.book_id) AS total_books
FROM publishers p
INNER JOIN books b
    ON p.publisher_id = b.publisher_id
WHERE b.num_pages > 50
GROUP BY p.publisher
ORDER BY total_books DESC
LIMIT 1;
"""

top_publisher = pd.read_sql(query, engine)

top_publisher

,publisher,total_books
0,Penguin Books,42


Conclusión

In [32]:
print(
    f"La editorial con más libros mayores a 50 páginas es "
    f"{top_publisher.loc[0,'publisher']}."
)

La editorial con más libros mayores a 50 páginas es Penguin Books.


Esta editorial podría ser un socio estratégico importante debido al volumen de publicaciones relevantes.

### Objetivo 4
Autor con la mayor calificación promedio (considerando solo libros con al menos 50 valoraciones)
### Consulta SQL

In [33]:
query = """
WITH books_50_ratings AS (

    SELECT
        book_id,
        AVG(rating) AS avg_rating,
        COUNT(rating_id) AS rating_count

    FROM ratings

    GROUP BY book_id

    HAVING COUNT(rating_id) >= 50
)

SELECT
    a.author,
    ROUND(AVG(b50.avg_rating),2) AS average_author_rating

FROM books_50_ratings b50

INNER JOIN books b
    ON b50.book_id = b.book_id

INNER JOIN authors a
    ON b.author_id = a.author_id

GROUP BY a.author

ORDER BY average_author_rating DESC

LIMIT 1;
"""

top_author = pd.read_sql(query, engine)

top_author

,author,average_author_rating
0,J.K. Rowling/Mary GrandPré,4.28


Conclusión

In [34]:
print(
    f"El autor mejor valorado es "
    f"{top_author.loc[0,'author']}." 
)

El autor mejor valorado es J.K. Rowling/Mary GrandPré.


Este autor representa una oportunidad importante para recomendaciones y promoción dentro de la plataforma.

## Objetivo 5
Número promedio de reseñas escritas por usuarios que calificaron más de 50 libros
### Consulta SQL

In [35]:
query = """
WITH active_users AS (

    SELECT
        username
    FROM ratings
    GROUP BY username
    HAVING COUNT(DISTINCT book_id) > 50
),

user_reviews AS (

    SELECT
        username,
        COUNT(review_id) AS review_count
    FROM reviews
    GROUP BY username
)

SELECT
    ROUND(AVG(COALESCE(review_count,0)),2) AS average_reviews
FROM active_users au
LEFT JOIN user_reviews ur
    ON au.username = ur.username;
"""

avg_reviews = pd.read_sql(query, engine)

avg_reviews

,average_reviews
0,24.33


Conclusión

In [36]:
print(
    f"Los usuarios que calificaron más de 50 libros "
    f"escribieron en promedio "
    f"{avg_reviews.iloc[0,0]} reseñas."
)

Los usuarios que calificaron más de 50 libros escribieron en promedio 24.33 reseñas.


Esto permite entender el nivel de participación de los usuarios más activos y determinar si existe una relación entre calificar libros y escribir reseñas.

## Conclusión General

Se identificó la cantidad de libros publicados después del año 2000, lo que permite evaluar la relevancia del catálogo moderno.

2. Se calculó el volumen de reseñas y la valoración promedio de cada libro para detectar títulos populares y mejor valorados.

3. Se identificó la editorial con la mayor cantidad de publicaciones relevantes (>50 páginas), útil para posibles alianzas estratégicas.

4. Se encontró el autor con mejor valoración promedio entre libros con suficiente evidencia estadística (50+ valoraciones).

5. Se analizó el comportamiento de los usuarios más activos para medir su nivel de interacción mediante reseñas.

Los resultados obtenidos pueden utilizarse para diseñar sistemas de recomendación, estrategias de marketing y propuestas de valor para una nueva plataforma de lectura digital.